# L02 · 언어모델은 어떻게 확률을 내는가

## Goal

- causal next-token objective를 설명한다
- token·sequence log-prob shape을 추적한다
- prompt·response·pad mask를 구분한다

## Setup

이 cell은 CPU·seed·offline 상태와 split hash를 먼저 고정합니다. toy 연산은 결정론적인 CPU 연산만 쓰며, package trainer의 전역 결정론 기본값은 유지합니다.

In [1]:
import hashlib, json, os, platform, random, sys
from pathlib import Path
os.environ.setdefault("TORCH_DEVICE_BACKEND_AUTOLOAD", "0")
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file()), None)
if ROOT is None:
    raise RuntimeError("Run this notebook inside the RL-study repository")
sys.path.insert(0, str(ROOT / "src"))
import torch
from rl_study import __version__
from rl_study.data import build_tiny_reasoning
from rl_study.runtime import resolve_device, seed_everything
# These notebooks use only deterministic CPU toy kernels.  PyTorch 2.13's global
# guard imports the full Inductor stack, so keep the package's strict default for
# trainers while avoiding that unrelated startup cost in fresh teaching kernels.
seed_everything(42, deterministic=False)
random.seed(42)
language = os.environ.get("RL_STUDY_NOTEBOOK_LANGUAGE", "ko")
resolution = resolve_device("cpu")
dataset = build_tiny_reasoning(seed=42)
config_hash = "sha256:" + hashlib.sha256(b"L02:toy:42").hexdigest()
print(f"lesson=L02 language={language} profile=toy")
print("seed=42 network_required=False deterministic_scope=seeded_cpu_toy")
print(f"python={platform.python_version()} rl_study={__version__} torch={torch.__version__}")
print(f"requested_device=cpu resolved_device={resolution.resolved} fallback_used={resolution.fallback_used}")
print(f"config_hash={config_hash} data_split_hash={dataset.split_hash}")

lesson=L02 language=ko profile=toy
seed=42 network_required=False deterministic_scope=seeded_cpu_toy
python=3.10.12 rl_study=0.1.0.dev0 torch=2.13.0
requested_device=cpu resolved_device=cpu fallback_used=False
config_hash=sha256:709e8a5c9d82a72fe43146d65d9657eea17509dfd65e9e3d5d1f363ee173a420 data_split_hash=sha256:f238657bbf6c0a112debf7ef3ffafb452c14308dfb5ce57d9abe4f77ac1deedd


## Steps

### 1. 현재 위치와 핵심 식

⏱ 5분 · 1/3 section · [필수/CORE]

현재 위치: 확률·미분 → **causal LM과 token mask** → LLM policy

$$\log\pi_\theta(y\mid x)=\sum_t m_t\log p_\theta(y_t\mid x,y_{<t})$$

causal LM logits는 `[batch, time, vocabulary]`이고 target은 한 칸 왼쪽 logits와 맞춥니다. prompt는 조건이며 action이 아니므로 policy loss에는 response와 EOS 위치만 남깁니다. padding은 계산량을 맞출 뿐 reward를 받지 않습니다.

### 2. 작은 숫자로 실행

⏱ 6분 · 2/3 section · [필수/CORE]

**먼저 예측:** 문자열 `2`가 response일 때 action mask 합은 1일까요, EOS까지 포함한 2일까요? 20초 동안 답을 적은 뒤 실행하세요.

<details><summary>정답 보기</summary>이 저장소의 계약은 생성 종료를 action으로 포함하므로 2입니다. 이 선택은 rollout과 update에서 동일해야 합니다.</details>

In [2]:
from rl_study.models import TinyCausalLM, TinyTokenizer, build_sequence_batch
from rl_study.models.sequence import response_sequence_log_probs
tokenizer = TinyTokenizer()
model = TinyCausalLM()
sequence_batch = build_sequence_batch(
    ["Q:1+1="], ["2"], tokenizer=tokenizer, max_length=64
)
sequence_logp = response_sequence_log_probs(model, sequence_batch)
print({"input_shape": list(sequence_batch.input_ids.shape),
       "target_shape": list(sequence_batch.action_mask.shape),
       "action_tokens": int(sequence_batch.action_mask.sum()),
       "sequence_logp": round(float(sequence_logp[0].detach()), 3)})

{'input_shape': [1, 9], 'target_shape': [1, 8], 'action_tokens': 2, 'sequence_logp': -149.19}


### 3. 구현 해부

⏱ 6분 · 3/3 section · [심화/DEEP DIVE]

**왜 이렇게 구현했나:** 평균 token log-prob와 합계 sequence log-prob는 길이에 대한 의미가 다릅니다. package API가 mask와 reduction을 명시해 DPO·PPO·GRPO가 서로 다른 암묵적 규칙을 갖지 않게 합니다.

**흔한 함정:** prompt target까지 합치면 긴 prompt가 update를 지배합니다. prompt/action mask의 교집합이 비어 있다는 truth-table 검사가 이를 잡습니다. 회귀 test: `test_prompt_and_action_mask_truth_table`.

**쉬어가기:** 지금 출력한 한 값만 설명할 수 있으면 다음 cell로 가세요.

## Checks

In [3]:
assert not bool((sequence_batch.prompt_target_mask & sequence_batch.action_mask).any())
assert int(sequence_batch.action_mask.sum()) == 2
print("checks=passed")

checks=passed


**회상 문제:** EOS를 mask에서 빼면 어떤 행동의 학습 신호가 사라지나요? 1~2문장으로 답하세요.

## 내가 자주 틀리는 것

- loss가 유한하면 구현도 맞다고 생각한다.
- `terminated`와 `truncated`, prompt와 action을 합친다.
- 한 seed의 작은 결과를 알고리즘 순위로 확대한다.

## 60초 요약

- **실행 결론:** 출력은 input 길이 9에 대해 예측 위치 8개, 실제 action 위치 2개를 보여 줍니다. sequence log-prob는 그 두 위치만 합친 값입니다.
- 실제 확인: `test_prompt_and_action_mask_truth_table`.
- 출력은 고정 seed의 toy 실행이며 논문 규모 결과가 아닙니다.

## Next Steps

1. L03에서 sequence를 잠시 내려놓고 exploration과 sampled reward의 가장 작은 실험을 만듭니다.
2. `[필수/CORE]` assertion을 한 번 깨뜨리고 오류를 읽습니다.
3. package test를 열어 notebook의 작은 식과 production guard를 연결합니다.

## Sources

- `instructgpt-2022` — `docs/sources.yml`